### Setup and Installations ###

In [ ]:
# Setup tools to avoid package conflicts
!pip install -U pip setuptools
# Install all necassary packages from shell file
!bash install_packages.sh
!pip install --upgrade transformers


### Training Script Start ###

In [ ]:
# StarCoder2-7B Fine-tuning Notebook
# ==================================
import os
import torch
import transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from trl import SFTTrainer
import bitsandbytes as bnb
from datetime import datetime

# Configuration
MODEL_NAME = "bigcode/starcoder2-7b"
DATASET_PATH = "jsonl_for_starCoder.jsonl"
OUTPUT_DIR = "starcoder2_finetuned"

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")

In [ ]:
# Wandb logging setup
import wandb

# Log in to wandb (only required once per environment/session)
wandb.login()

# Initialize wandb for this training run
wandb.init(
    project="starCoder-rpi-finetune",
    name="starCoder-lora-run-v1",  # change the run name if needed
    reinit=True,
    config={
        "batch_size": 2,
        "learning_rate": 2e-4,
        "dataset": "jsonl_for_starCoder.jsonl",
        "max_epochs": 3,
        "gradient_accumulation_steps": 4,
    },
)

In [ ]:
# 1. Load & prepare dataset
# =========================

def prepare_dataset():
    # Load dataset
    dataset = load_dataset('json', data_files=DATASET_PATH)['train']
    print(f"Loaded dataset with {len(dataset)} examples")
    
    # Convert prompt/completion pairs to StarCoder2 format
    def format_example(example):
        # Using code formatting rather than instruction formatting
        # since StarCoder2 is not an instruction model
        return {
            "text": f"# Raspberry Pi Project\n# Task: {example['prompt']}\n\n{example['completion']}"
        }
    
    formatted_dataset = dataset.map(format_example)
    splits = formatted_dataset.train_test_split(test_size=0.05)
    
    print(f"Training examples: {len(splits['train'])}")
    print(f"Validation examples: {len(splits['test'])}")
    
    # Show a sample
    print("\nSample example:")
    print(splits['train'][0]['text'][:200] + "...")
    
    return splits['train'], splits['test']

train_dataset, eval_dataset = prepare_dataset()

In [ ]:
# 2. Load tokenizer and model
# ==========================

# Initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model with quantization
print("Loading StarCoder2-7B with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

In [ ]:
# 3. Configure LoRA
# ================

# Set target modules for StarCoder2 architecture
target_modules = [
    "q_proj", 
    "k_proj", 
    "v_proj", 
    "o_proj", 
    "gate_proj", 
    "up_proj", 
    "down_proj"
]

# LoRA configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA to model
# Apply LoRA
model = get_peft_model(model, lora_config)

# ✅ Enable gradient checkpointing (reduces memory usage)
model.gradient_checkpointing_enable()

# ✅ Ensure inputs require gradients (needed for LoRA)
model.enable_input_require_grads()

# Print trainable vs total parameters for the model
model.print_trainable_parameters()

In [ ]:
# 4. Configure Training - Fixed Version
# ====================

# Import the correct TrainingArguments class
from transformers import TrainingArguments

# Set up training arguments with correct parameter names
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_torch",  # Use standard adamw as fused might not be available
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=True,  # Use bfloat16 precision
    logging_steps=10,
    # Fix for evaluation_strategy parameter
    eval_strategy="epoch",  # Try this instead of evaluation_strategy
    # Alternative: remove evaluation_strategy completely if it's still causing issues
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="wandb",
    run_name="starcoder2-finetune-rpi"
)

# Initialize SFT Trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=8096,
)

In [ ]:
# 5. Train the model
# =================
import json
from dataclasses import asdict

print("Starting training...")
train_result = trainer.train()
print(f"Training completed with metrics: {train_result.metrics}")

# Save adapter
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
final_model_path = os.path.join(OUTPUT_DIR, f"starcoder2_raspberrypi_{timestamp}")
trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)


with open(os.path.join(final_model_path, "training_config.json"), "w") as f:
    json.dump(training_args.to_dict(), f, indent=4)

with open(os.path.join(final_model_path, "lora_config.json"), "w") as f:
    json.dump(asdict(lora_config), f, indent=4)

print(f"Model adapter saved to {final_model_path}")

### Single Training Script ###

In [ ]:
import os
from datetime import datetime
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer  # pip install trl

# =========================
# 0. Assumes train_dataset and eval_dataset are already loaded/prepared
# =========================

# Set your model name and output directory here
MODEL_NAME = "bigcode/starcoder2-7b"
OUTPUT_DIR = "./results/starcoder2_rpi"

# ==========================
# 1. Load tokenizer & model
# ==========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading StarCoder2-7B with 4-bit quantization...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# =================================
# 2. Check for latest checkpoint
# =================================

def get_latest_checkpoint(output_dir):
    """Returns path to latest checkpoint, or None if not found."""
    checkpoints = []
    if not os.path.isdir(output_dir):
        return None
    for subdir in os.listdir(output_dir):
        if subdir.startswith("checkpoint-"):
            full = os.path.join(output_dir, subdir)
            if os.path.isdir(full):
                checkpoints.append((full, os.path.getmtime(full)))
    if checkpoints:
        checkpoints.sort(key=lambda x: x[1])
        return checkpoints[-1][0]
    return None

os.makedirs(OUTPUT_DIR, exist_ok=True)
logdir = os.path.join(OUTPUT_DIR, "logs")
os.makedirs(logdir, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
tb_dir = os.path.join(OUTPUT_DIR, "tensorboard", timestamp)
os.makedirs(tb_dir, exist_ok=True)

latest_ckpt = get_latest_checkpoint(OUTPUT_DIR)
if latest_ckpt:
    print(f"Resuming from latest checkpoint: {latest_ckpt}")
else:
    print("No checkpoint found, starting from scratch.")

# ===============================
# 3. Configure TrainingArguments
# ===============================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    optim="adamw_torch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    bf16=True,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    report_to="tensorboard",    # only tensorboard supported
    logging_dir=logdir,
    disable_tqdm=True,
    dataloader_num_workers=2,
    run_name=f"starcoder2_rpi_{timestamp}",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

# ================================
# 4. Initialize SFT Trainer
# ================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=8096,
)

# ================================
# 5. Train with resume support
# ================================

print("Starting training...")
train_result = trainer.train(resume_from_checkpoint=latest_ckpt if latest_ckpt else None)
print(f"Training completed with metrics: {train_result.metrics}")

# ================================
# 6. Save model, tokenizer, logs
# ================================

final_model_path = os.path.join(OUTPUT_DIR, f"starcoder2_raspberrypi_{timestamp}")
trainer.model.save_pretrained(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"Model adapter saved to {final_model_path}")

# Save full log history as CSV
import pandas as pd
pd.DataFrame(trainer.state.log_history).to_csv(
    os.path.join(OUTPUT_DIR, f"full_log_history_{timestamp}.csv"), index=False
)

# Save summary metrics as JSON
import json
with open(os.path.join(OUTPUT_DIR, f"metrics_{timestamp}.json"), "w") as f:
    json.dump(train_result.metrics, f, indent=2)
print(f"Training logs and metrics saved to {OUTPUT_DIR}")

# ================================
# 7. TensorBoard instructions
# ================================
print(f"\nTo view logs, run in terminal:\n  tensorboard --logdir {tb_dir}\n")
print(f"All logs (csv, metrics, TensorBoard) are saved in {OUTPUT_DIR} for your report.")

### Batch Inference Script ###

In [ ]:
#!/usr/bin/env python3
"""
Generate and Compare Responses from Base and Fine-Tuned DeepSeek Models

This script:
1. Loads prompts from a JSON file
2. Generates code responses using both base and fine-tuned DeepSeek Coder models
3. Saves results to:
   - Generates Individual C files (one per prompt)
   - A consolidated JSON file with all results

Created: 2025-05-18 02:44:17
Author: Ali Hassan
"""
import json
import os
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, AutoPeftModelForCausalLM
import re
from datetime import datetime
import getpass
from tqdm import tqdm

# Configuration
INPUT_JSON = "Test_Train_Samples_Evaluation_Set2_17May2025.json"
OUTPUT_JSON = "starcoder_responses_eval2_17May2025.json"
OUTPUT_DIR = "starcoder_generated_code"
BASE_MODEL_ID = "bigcode/starcoder2-7b"
FINETUNED_MODEL_PATH = "./results/starcoder2_rpi/starcoder2_raspberrypi_20250423_204222"
RANDOM_SEED = 42
TIMESTAMP = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
USERNAME = "Ali Hassan"
EXECUTION_DEVICE = "GPU" if torch.cuda.is_available() else "CPU"

def setup_directory(directory):
    """Create output directory if it doesn't exist."""
    os.makedirs(directory, exist_ok=True)
    print(f"Output directory set up: {directory}")

def load_json(file_path):
    """Load examples from a JSON file containing a list of objects."""
    print(f"Loading data from {file_path}...")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Input file not found: {file_path}")

    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            examples = json.load(f)
            if not isinstance(examples, list):
                raise ValueError("Expected a list of JSON objects in the file.")
        except json.JSONDecodeError as e:
            raise ValueError(f"Error parsing JSON: {e}")

    print(f"Successfully loaded {len(examples)} examples.")
    return examples

def load_models():
    """Load both the base and fine-tuned StarCoder2 models."""
    print("\n1. Loading base StarCoder2 model...")
    start_time = time.time()
    
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    # StarCoder2 already has a pad token set up correctly
    if base_tokenizer.pad_token is None:
        base_tokenizer.pad_token = base_tokenizer.eos_token
    
    print(f"   Base model loaded in {time.time() - start_time:.2f} seconds")
    
    print("\n2. Loading fine-tuned StarCoder2 model...")
    start_time = time.time()
    
    # Load the base model first
    finetuned_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    # Then load the PEFT/LoRA adapter on top
    finetuned_model = PeftModel.from_pretrained(
        finetuned_base,
        FINETUNED_MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    finetuned_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    if finetuned_tokenizer.pad_token is None:
        finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token
    
    print(f"   Fine-tuned model loaded in {time.time() - start_time:.2f} seconds")
    
    return (base_model, base_tokenizer), (finetuned_model, finetuned_tokenizer)

def extract_clean_code(response):
    """Extract clean C code from a model response, removing explanatory text."""
    
    # --- Extract C code block ---
    code_match = re.search(r"```c\n(.*?)```", response, re.DOTALL)
    if not code_match:
        code_match = re.search(r"```(.*?)```", response, re.DOTALL)
    if code_match:
        return code_match.group(1).strip()

    # --- Fallback: Extract code based on C syntax patterns ---
    lines = response.split('\n')
    code_lines = []
    in_code = False

    explanation_patterns = [
        r'^(Here\'s|This|Let\'s|Now|First|The code|I\'ll|To|We|That\'s|This will)',
        r'.*\b(explains|provides|shows|demonstrates|illustrates)\b.*',
        r'^In this code',
        r'^This program',
    ]

    for line in lines:
        stripped = line.strip()

        # Skip empty or explanatory lines unless already in code
        if not stripped and not in_code:
            continue
        if any(re.match(pattern, stripped, re.IGNORECASE) for pattern in explanation_patterns) and not in_code:
            continue

        # Detect C code patterns
        if re.match(r'^(#include|int\s+main|void\s+main|void\s+\w+|int\s+\w+|float\s+\w+|char\s+\w+|printf|scanf|for\s*\(|while\s*\(|if\s*\(|return\b)', stripped):
            in_code = True
            code_lines.append(line)
        elif in_code or (stripped and not stripped.endswith('.') and not stripped.endswith('?')):
            code_lines.append(line)

    if code_lines:
        return '\n'.join(code_lines)

    return response

def generate_response(model, tokenizer, prompt, cal_max_new_tokens = 2048):
    """Generate a response from a StarCoder2 model."""
    # StarCoder2 doesn't use the same instruction format as Mistral
    # We'll use a simple prompt format
    formatted_prompt = f"You are a Raspberry Pi C Code Expert. Write a C code for Raspberry Pi that accomplishes the following task: {prompt}\n\n"
    
    # Tokenize and ensure we're not exceeding any length limits
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    input_length = len(inputs["input_ids"][0])
    
    print(f"   Input length: {input_length} tokens")
    
    with torch.no_grad():
        start_time = time.time()
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=cal_max_new_tokens,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id
        )
        
        generation_time = time.time() - start_time
    
    response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    clean_code = extract_clean_code(response)
    
    print(f"   Generation time: {generation_time:.2f} seconds")
    return response, clean_code, generation_time

def save_to_file(example_id, prompt, source, base_response, finetuned_response, output_dir):
    """Save responses into three separate C files."""
    # File paths
    combined_file = os.path.join(output_dir, f"example_combinedStarCoder_id_{example_id}_{source}.c")
    base_file = os.path.join(output_dir, f"example_baseStarCoder_id_{example_id}_{source}.c")
    finetuned_file = os.path.join(output_dir, f"example_fineTunedStarCoder_id_{example_id}_{source}.c")

    # --- Write combined file ---
    with open(combined_file, "w", encoding="utf-8") as f:
        f.write(f"""\
/*
 * Generated StarCoder2 responses for prompt ID: {example_id}
 * Source: {source}
 * Generated on: {TIMESTAMP}
 * Generated by: {USERNAME}
 */

/* Original prompt:
{prompt}
*/

/* -----------------------------------------
   Base StarCoder2 Response ({BASE_MODEL_ID}):
   ----------------------------------------- */
{base_response}

/* -----------------------------------------
   Fine-Tuned StarCoder2 Response:
   ----------------------------------------- */
{finetuned_response}
""")

    # --- Write base-only file ---
    with open(base_file, "w", encoding="utf-8") as f:
        f.write(f"""\
/*
 * Base StarCoder2 Response for prompt ID: {example_id}
 * Source: {source}
 * Generated on: {TIMESTAMP}
 * Generated by: {USERNAME}
 */

/* Original prompt:
{prompt}
*/

{base_response}
""")

    # --- Write finetuned-only file ---
    with open(finetuned_file, "w", encoding="utf-8") as f:
        f.write(f"""\
/*
 * Fine-Tuned StarCoder2 Response for prompt ID: {example_id}
 * Source: {source}
 * Generated on: {TIMESTAMP}
 * Generated by: {USERNAME}
 */

/* Original prompt:
{prompt}
*/

{finetuned_response}
""")

    return combined_file, base_file, finetuned_file

def main():
    """Main execution function with checkpointing."""
    print(f"=== StarCoder2 Response Generation ===")
    print(f"Date/Time: {TIMESTAMP}")
    print(f"User: {USERNAME}")

    torch.manual_seed(RANDOM_SEED)
    setup_directory(OUTPUT_DIR)

    examples = load_json(INPUT_JSON)
    total_examples = len(examples)

    # Load checkpoint if exists
    completed_results = []
    if os.path.exists(OUTPUT_JSON):
        try:
            with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
                completed_results = json.load(f)
                print(f"🔁 Resuming from checkpoint. Already processed: {len(completed_results)} examples.")
        except Exception as e:
            print(f"⚠️ Error reading checkpoint: {e}")
            completed_results = []

    start_index = len(completed_results)
    (base_model, base_tokenizer), (finetuned_model, finetuned_tokenizer) = load_models()

    start_time = time.time()

    for i, example in enumerate(tqdm(examples[start_index:], desc="🧠 Generating", unit="example", ncols=100), start=start_index):
        example_id = example["id"]
        prompt = example["input"]
        source = example["data_source"]
        code_reference = example["output"]

        output_len = len(code_reference)
        estimated_tokens = int((output_len / 4) * 2) if output_len > 0 else 2048
        cal_max_new_tokens = min(8192, max(128, estimated_tokens))

        tqdm.write(f"\n[{i+1}/{total_examples}] Processing prompt {example_id} (Source: {source}) (Allowed Tokens: {cal_max_new_tokens})")

        tqdm.write(f"   Generating base DeepSeek response...")
        base_response, base_code, base_time = generate_response(base_model, base_tokenizer, prompt, cal_max_new_tokens)

        tqdm.write(f"   Generating fine-tuned DeepSeek response...")
        finetuned_response, finetuned_code, finetuned_time = generate_response(finetuned_model, finetuned_tokenizer, prompt, cal_max_new_tokens)

        save_to_file(example_id, prompt, source, base_code, finetuned_code, OUTPUT_DIR)
        tqdm.write(f"   Saved to file set")

        example["base_response"] = base_response
        example["base_code"] = base_code
        example["base_execution_time"] = base_time
        example["finetuned_response"] = finetuned_response
        example["finetuned_code"] = finetuned_code
        example["finetuned_execution_time"] = finetuned_time
        example["allowed_max_new_tokens"] = cal_max_new_tokens

        completed_results.append(example)

        # Save checkpoint
        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(completed_results, f, indent=2)

        elapsed = time.time() - start_time
        avg_time = elapsed / (i + 1)
        remaining = avg_time * (total_examples - i - 1)
        eta_min, eta_sec = divmod(int(remaining), 60)
        tqdm.write(f"⏱️ Time/iter: {avg_time:.2f}s | ETA: {eta_min}m {eta_sec}s")

    print("\n=== StarCoder2 Generation Complete ===")
    print(f"- Processed {total_examples} prompts")
    print(f"- C files saved to: {OUTPUT_DIR}/")
    print(f"- Results saved to: {OUTPUT_JSON}")
    print(f"- Timestamp: {TIMESTAMP}")
    print(f"- Generated by: {USERNAME}")
   
if __name__ == "__main__":
    main()